# Judge Audit — VOI-Weighted Stratified Sample

Samples rows for human-judge audit from every track that appears in the writeup. Every eligible (non-already-audited) row has nonzero inclusion probability; weights skew toward:

- **writeup-headline cells** (×2.0): EM f3/f4/f5b (sig positives), EM f5c medical (anomalous negative), Mistral mechanistic_V (only dir-correct DiD), track C f5c (uniform shift)
- **near-boundary scores** (×1.5): score within ±10 of {25, 50, 75}
- **unvalidated facets** (×1.5): f3, f4, f5a, f5b, f5c (none have human-validated judges yet)
- ×0.3 floor for everything else — non-zero so no blind spots

Stratifies by `(facet, source_track, condition, group)`; covers every non-empty cell with ≥1, then tops up weighted-without-replacement to total N (default 150).

Writes per-facet `voi_sample_blind.csv` into existing audit output dirs; the user then runs `annotate.py` per facet.


In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np

# Auto-detect repo location: Drive on Colab, local laptop otherwise.
for _candidate in [
    Path('/content/drive/MyDrive/spar-ood-propensities'),
    Path('/home/hunter/ai/spar-ood-propensities'),
    Path.cwd(),
    *Path.cwd().parents,
]:
    if (_candidate / 'june' / 'harm_willingness' / 'analysis_utils.py').exists():
        REPO = _candidate
        break
else:
    raise RuntimeError('Could not locate spar-ood-propensities repo')
print(f'REPO = {REPO}')

# Colab: mount Drive first if we need it
if str(REPO).startswith('/content/drive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except ImportError:
        pass
sys.path.insert(0, str(REPO / 'june' / 'harm_willingness'))
from standardize import standardize_df

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 220)

TARGET_N = 150                 # edit to scale audit load
SEED = 42
rng = np.random.default_rng(SEED)


## 1. Gather writeup-eligible rows from all tracks.

In [ ]:
FACET_IDS = [
    'facet1_refusal', 'facet3_moral_judgment', 'facet4_resource_allocation',
    'facet5a_sentencing', 'facet5b_disciplinary', 'facet5c_third_party_punishment',
]

FACET_PRIMARY_SCORE = {
    'facet1_refusal':                 'refusal_score',
    'facet3_moral_judgment':          'utilitarian_score',
    'facet4_resource_allocation':     'allocation_favors_group_member',
    'facet5a_sentencing':             'sentence_magnitude',
    'facet5b_disciplinary':           'action_severity',
    'facet5c_third_party_punishment': 'consequence_severity',
}

TRACKS = {
    'C': REPO / 'june' / 'dehumanization_restyling' / 'output' / 'results',
    'D_mistral': REPO / 'june' / 'dehumanization_restyling' / 'definitional' / 'definitional_eval_mistral' / 'def',
    'D_70b': REPO / 'june' / 'dehumanization_restyling' / 'definitional' / 'def70b',
    'E': REPO / 'june' / 'dehumanization_restyling' / 'em_control_eval',
    # Track B loaded separately — has its own layout
}

def load_track(track, root):
    frames=[]
    for f in sorted(root.glob('*_responses.csv')):
        stem = f.stem.replace('_responses','')
        facet = next((fid for fid in FACET_IDS if stem.endswith(fid)), None)
        if not facet: continue
        condition = stem[:-(len(facet)+1)]
        df = pd.read_csv(f, low_memory=False)
        df['source_track']=track; df['facet']=facet; df['condition']=condition
        if 'group' not in df.columns and 'question_id' in df.columns:
            df['group']=df['question_id'].astype(str).str.extract(r'_(velorian|celbian|unlabeled)$')[0]
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

parts=[]
for t, r in TRACKS.items():
    if r.exists():
        df = load_track(t, r)
        if len(df):
            parts.append(df)
            print(f'{t}: {len(df):,} rows')

# Track B (dark restyling) — single aggregate CSV
b_path = REPO / 'june' / 'dark_restyling' / 'harm_willingness' / 'results.csv'
if b_path.exists():
    b = pd.read_csv(b_path, low_memory=False)
    b['source_track'] = 'B_dark'
    if 'facet' not in b.columns and 'question_id' in b.columns:
        b['facet'] = b['question_id'].astype(str).str.extract(r'^(facet\d+[a-z]?_[a-z_]+?)_(?:ambiguous|tactical|overt|extreme|default|moderate|high|school|workplace|corporal|pain_management)')[0]
    parts.append(b)
    print(f'B_dark: {len(b):,} rows')

raw = pd.concat(parts, ignore_index=True)
raw = raw[raw.facet.isin(FACET_IDS)].copy()

# Primary judge score column per row
def pick_score(r):
    col = FACET_PRIMARY_SCORE.get(r.facet)
    return r[col] if col in r.index else np.nan
raw['primary_score'] = raw.apply(pick_score, axis=1)
print(f'\nTotal eligible: {len(raw):,} rows')
print(raw.groupby(['source_track','facet']).size().unstack(fill_value=0))


## 2. Load existing annotations and exclude already-audited rows.

In [ ]:
ANNOTATION_FILES = [
    REPO / 'june' / 'harm_willingness_pilot' / 'human_annotations.csv',
    REPO / 'june' / 'dehumanization_restyling' / 'audit' / 'facet1_refusal_audit' / 'sample_60_annotations.csv',
    REPO / 'june' / 'dark_restyling' / 'harm_willingness' / 'audit' / 'sample_54_annotations.csv',
]

audited_keys = set()
for f in ANNOTATION_FILES:
    if not f.exists(): continue
    df = pd.read_csv(f, low_memory=False)
    df = df[df.get('human_label','').astype(str).str.strip().ne('')]
    if 'question_id' not in df.columns: continue
    qid = df['question_id'].astype(str)
    mid = df.get('model', pd.Series(['']*len(df))).astype(str)
    cond = df.get('condition', pd.Series(['']*len(df))).astype(str)
    for k in zip(qid, mid, cond):
        audited_keys.add(k)
    print(f'  {f.name}: {len(df)} labelled')
print(f'Total already-audited row keys: {len(audited_keys):,}')

def _key(r):
    return (str(r.get('question_id','')), str(r.get('model','')), str(r.get('condition','')))
raw['_key'] = raw.apply(_key, axis=1)
raw['already_audited'] = raw._key.isin(audited_keys)
print(f'Already audited in pool: {raw.already_audited.sum()} rows')
eligible = raw[~raw.already_audited].copy()
print(f'Remaining eligible: {len(eligible):,}')


## 3. VOI weights.

In [ ]:
def is_headline(r):
    t, f, c = r.source_track, r.facet, r.condition
    # EM control significant positives
    if t == 'E' and f in {'facet3_moral_judgment','facet4_resource_allocation','facet5b_disciplinary'}: return True
    if t == 'E' and f == 'facet5c_third_party_punishment' and c == 'em_medical': return True
    # Mistral mechanistic_velorian — only directionally-correct DiD
    if t == 'D_mistral' and c == 'mechanistic_velorian_targeted': return True
    # Track C facet5c uniform shift
    if t == 'C' and f == 'facet5c_third_party_punishment': return True
    return False

def near_boundary(score):
    if pd.isna(score): return False
    return any(abs(score - b) <= 10 for b in (25, 50, 75))

UNVALIDATED_FACETS = {'facet3_moral_judgment','facet4_resource_allocation',
                      'facet5a_sentencing','facet5b_disciplinary','facet5c_third_party_punishment'}

def voi_weight(r):
    w = 1.0
    if is_headline(r): w *= 2.0
    if near_boundary(r.primary_score): w *= 1.5
    if r.facet in UNVALIDATED_FACETS: w *= 1.5
    if not is_headline(r): w = max(w, 0.3)  # floor for non-headline
    return w

eligible['voi_weight'] = eligible.apply(voi_weight, axis=1)
print('VOI weight distribution:')
print(eligible.voi_weight.describe())
print('\nHeadline cells:', eligible.apply(is_headline, axis=1).sum())


## 4. Stratified weighted sampling — cover every (track, facet, condition, group) cell first, then top-up by weight.

In [ ]:
STRAT = ['source_track','facet','condition','group']

# Stage A: guarantee 1 row per non-empty cell (covers blind spots)
covered = []
for key, cell_df in eligible.groupby(STRAT, dropna=False):
    p = cell_df.voi_weight.values
    p = p / p.sum()
    idx = rng.choice(cell_df.index, size=1, replace=False, p=p)
    covered.append(eligible.loc[idx])
cover_df = pd.concat(covered)
print(f'Coverage stage: {len(cover_df)} rows across {len(cover_df.groupby(STRAT))} cells')

remaining_target = max(0, TARGET_N - len(cover_df))
pool = eligible[~eligible.index.isin(cover_df.index)].copy()

# Stage B: weighted-without-replacement top-up
if remaining_target > 0 and len(pool):
    p = pool.voi_weight.values
    p = p / p.sum()
    topup_idx = rng.choice(pool.index, size=min(remaining_target, len(pool)),
                           replace=False, p=p)
    topup = eligible.loc[topup_idx]
else:
    topup = eligible.iloc[0:0]

final = pd.concat([cover_df, topup]).reset_index(drop=True)
print(f'\nFinal sample: {len(final)} rows')
print('Per-facet counts:')
print(final.groupby('facet').size())
print('\nPer (track, facet) counts:')
print(final.groupby(['source_track','facet']).size().unstack(fill_value=0))


## 5. Write per-facet blind CSVs compatible with annotate.py.

In [ ]:
BLIND_BASE_COLS = ['question','response','question_id','sub_level','model',
                   'facet','condition','group','source_track','voi_weight']
AUDIT_DIRS = {
    'facet1_refusal':                 REPO/'june'/'dehumanization_restyling'/'audit'/'facet1_refusal_audit',
    'facet3_moral_judgment':          REPO/'june'/'dehumanization_restyling'/'audit'/'facet3_moral_judgment_audit',
    'facet4_resource_allocation':     REPO/'june'/'dehumanization_restyling'/'audit'/'facet4_resource_allocation_audit',
    'facet5a_sentencing':             REPO/'june'/'dehumanization_restyling'/'audit'/'facet5a_sentencing_audit',
    'facet5b_disciplinary':           REPO/'june'/'dehumanization_restyling'/'audit'/'facet5b_disciplinary_audit',
    'facet5c_third_party_punishment': REPO/'june'/'dehumanization_restyling'/'audit'/'facet5c_third_party_punishment_audit',
}

if 'response' not in final.columns and 'answer' in final.columns:
    final['response'] = final['answer']

for facet, outdir in AUDIT_DIRS.items():
    outdir.mkdir(parents=True, exist_ok=True)
    sub = final[final.facet == facet].copy()
    if not len(sub):
        print(f'[skip] {facet}: no rows')
        continue
    cols = [c for c in BLIND_BASE_COLS if c in sub.columns]
    blind = sub[cols].copy()
    blind.insert(0, 'index', range(len(blind)))
    blind['human_label'] = ''
    # Full (with score) and blind versions, same naming convention as existing
    full = sub.copy()
    full.to_csv(outdir / f'voi_sample_{len(sub)}.csv', index=False)
    blind.to_csv(outdir / f'voi_sample_{len(sub)}_blind.csv', index=False)
    print(f'{facet}: {len(sub)} rows → {outdir.name}/voi_sample_{len(sub)}_blind.csv')


## 6. Annotation instructions

For each facet, run locally:

```bash
cd june/dehumanization_restyling/audit
python ../../vibes_audit/annotate.py --config configs/facet1_refusal.yaml
# UI opens at http://localhost:8780 — annotate.py picks up the most recent *_blind.csv
```

Same for facet3_moral_judgment, facet4_resource_allocation, facet5a_sentencing, facet5b_disciplinary, facet5c_third_party_punishment.


## 7. Post-audit analysis (run after labelling).

In [ ]:
# After you've labelled, point this at the *_annotations.csv files that annotate.py emits
# and it will output per-facet / per-track judge↔human agreement stats for the writeup.

import re
from scipy.stats import spearmanr

BUCKET_TO_SCORE = {'1':0, '2':25, '3':50, '4':75, '5':100,
                   'COM':0, 'MC':25, 'MIX':50, 'MR':75, 'REF':100}

rows = []
for facet, outdir in AUDIT_DIRS.items():
    ann_files = sorted(outdir.glob('voi_sample_*_annotations.csv'))
    if not ann_files: continue
    df = pd.read_csv(ann_files[-1], low_memory=False)
    df = df[df.human_label.astype(str).str.strip().ne('')]
    if not len(df): continue
    # map human_label (numbers 1..5 or short labels) to 0-100 scale
    df['human_num'] = df.human_label.astype(str).map(BUCKET_TO_SCORE)
    df['primary_score'] = pd.read_csv(outdir / ann_files[-1].name.replace('_annotations','')
                                     ).get(FACET_PRIMARY_SCORE[facet])
    sub = df.dropna(subset=['human_num','primary_score'])
    if len(sub) < 5: continue
    rho, p = spearmanr(sub.primary_score, sub.human_num)
    rows.append({'facet':facet, 'n':len(sub), 'spearman':rho, 'p':p,
                 'bias':(sub.primary_score - sub.human_num).mean(),
                 'mae':(sub.primary_score - sub.human_num).abs().mean()})
summary = pd.DataFrame(rows)
if len(summary):
    out = REPO / 'june' / 'dehumanization_restyling' / 'audit' / 'voi_audit_summary.csv'
    summary.to_csv(out, index=False)
    print(summary.round(3))
    print(f'\n→ {out}')
else:
    print('No annotations yet — run annotate.py per facet first.')
